In [1]:
import sys
import os
# Add build directory to path
# build_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "python_lib")
# sys.path.append(build_path)

from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
import logging
from datetime import datetime
import os
from logging import FileHandler
import tqdm
from collections import deque
from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.mixed_node import MixedNode, SplitNode
import datetime
import matplotlib.pyplot as plt
from blackjack.game_tree_to_json import game_tree_to_json
from blackjack.tree_utils import iterate_nodes_by_levels
import os
import pickle

In [2]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [3]:
from collections import defaultdict
from collections import Counter
from blackjack.abstract_node import ValueNode
from blackjack.mixed_node import DecisionNode



def get_nodes_by_levels(root_node, nodes_by_level_dict=None):
    if nodes_by_level_dict is None:
        nodes_by_level_dict = defaultdict(list)
    queue = deque()
    queue.append((root_node, 0))
    while queue:
        node, node_level = queue.popleft()
        nodes_by_level_dict[node_level].append(node)
        for child in node.children:
            queue.append((child, node_level + 1))
    return nodes_by_level_dict


def log_tree_structure(root_node):
    nodes_dict = get_nodes_by_levels(root_node)
    print(f"Total: {sum(len(nodes) for nodes in nodes_dict.values())} nodes")
    for lvl, nodes in nodes_dict.items():
        stage_count = count_stages(nodes)
        print(f"Level {lvl}: {len(nodes)} nodes", stage_count)


def count_stages(node_list):
    stage_counter = Counter()
    for node in node_list:
        stage_name = node.__class__.__name__

        if hasattr(node, "bj_round") and node.bj_round is not None:
            stage = node.bj_round.get_stage()
            if stage == BJStage.PLAYER_CARD:
                if len(node.children) == 1:
                    stage_name += "_PLAYER_CARD_SINGLE"
                elif len(node.children) < 10:
                    stage_name += "_PLAYER_CARD_PARTIAL"
                else:
                    stage_name += "_PLAYER_CARD_FULL"
            elif isinstance(node, DecisionNode):
                stage_name += "_PENDING" if node.decision_choice is None else "_DECIDED"
            else:
                stage_name += "_" + stage.name
        
        stage_counter[stage_name] += 1
    return stage_counter

In [4]:
main_actions_list = [
    PlayerAction.HIT,
    PlayerAction.STAND,
    PlayerAction.DOUBLE,
    PlayerAction.SPLIT
]


def best_action_hard_vs_dealer(hard_value, dealer_card):
    if hard_value < 5 or hard_value > 19:
        raise ValueError("Invalid hard value")

    if hard_value >= 12:
        player_card_0 = 10
    else:
        player_card_0 = 2
    
    player_card_1 = hard_value - player_card_0 

    cards = [
        Card(Rank.from_value(player_card_0)),
        Card(Rank.from_value(player_card_1)),
        Card(Rank.from_value(dealer_card))  
    ]
    
    bj_round = BJRound(rules)
    shoe = ProbabilisticRankShoe(8)
    bj_round.start_round(10)

    bj_round.take_card(cards[0])
    bj_round.take_card(cards[1])
    bj_round.take_card(cards[2])

    shoe.burn_rank_value(cards[0].rank_value())
    shoe.burn_rank_value(cards[1].rank_value())
    shoe.burn_rank_value(cards[2].rank_value())

    root_node = MixedNode(bj_round, shoe, monte_carlo_depth=6)
    root_node.build_tree()
    
    expected_value_0 = root_node.get_value()
    expected_value_1 = None

    main_action = None
    insurance_action = None
    action_node = root_node
    while main_action is None:
        stage = action_node.bj_round.get_stage()
        if stage == BJStage.DEALER_CHECK_BJ:
            # find branch where dealer doesn't have bj
            for ch in action_node.children:
                if ch.last_action == DealerAction.CONFIRM_NO_BLACKJACK:
                    action_node = ch
                    break
        elif stage == BJStage.PLAYER_OFFERED_INSURANCE:
            child_idx = np.argmax(action_node.children_prob)
            action_node = action_node.children[child_idx]
            insurance_action = action_node.last_action
        elif stage == BJStage.PLAYER_ACTION:
            action_child_idx = np.argmax(action_node.children_prob)
            after_action_node = action_node.children[action_child_idx]
            main_action = after_action_node.bj_round.last_action
            expected_value_1 = after_action_node.get_value()
        else:
            raise RuntimeError(f"Unexpected stage {stage}")
    
    return main_action, insurance_action, expected_value_0, expected_value_1

In [5]:
# action_data = {}

# for hard in tqdm.tqdm(range(5, 20)):
#     for dealer in range(2, 12):
#         result = best_action_hard_vs_dealer(hard, dealer)
#         action_data[(hard, dealer)] = result

# actions_only = {k: v[0] for k, v in action_data.items()}

# out_dir = "logs"
# out_path = os.path.join(out_dir, f"action_data.pkl")
# with open(out_path, "wb") as f:
#     pickle.dump(action_data, f, protocol=pickle.HIGHEST_PROTOCOL)

# actions_only_path = os.path.join(out_dir, f"actions_only.pkl")
# with open(actions_only_path, "wb") as f:
#     pickle.dump(actions_only, f, protocol=pickle.HIGHEST_PROTOCOL)


In [6]:
from blackjack.shoe import seed_shoe_rng

seed_shoe_rng(42)

bj_round = BJRound(rules)
shoe = ProbabilisticRankShoe(8)
bj_round.start_round(10)

cards = [
    Card(Rank.ACE),
    Card(Rank.ACE),
    Card(Rank.SIX) 
]

# cards = [
#     Card(Rank.TEN),
#     Card(Rank.SIX),
#     Card(Rank.ACE)  
# ]

cards = [c.rank_value() for c in cards]

bj_round.take_card(cards[0])
bj_round.take_card(cards[1])
bj_round.take_card(cards[2])

shoe.burn_rank_value(cards[0])
shoe.burn_rank_value(cards[1])
shoe.burn_rank_value(cards[2])

root_node = MixedNode(bj_round, shoe, max_hand_size_full_enum=3, player_card_initial_samples=2)

print(str(bj_round))

Last card: 6
Dealer 6,X
Player A,A(2/12)[$10]


In [7]:
for i in range(50):
    t0 = time.time()
    root_node.build_tree_layer(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)
    dt_whole = datetime.timedelta(seconds=seconds)
    print(f"Depth {i} built in {dt_whole} ({seconds} s)")
    if root_node.tree_completed():
        print("Tree completed")
        break

Depth 0 built in 0:00:00 (0.0 s)
Depth 1 built in 0:00:00 (0.0 s)
Depth 2 built in 0:00:00 (0.0 s)
Depth 3 built in 0:00:00 (0.0 s)
Depth 4 built in 0:00:01 (1.0 s)
Depth 5 built in 0:00:00 (0.0 s)
Depth 6 built in 0:00:00 (0.0 s)
Depth 7 built in 0:00:00 (0.0 s)
Depth 8 built in 0:00:00 (0.0 s)
Depth 9 built in 0:00:00 (0.0 s)
Depth 10 built in 0:00:00 (0.0 s)
Depth 11 built in 0:00:00 (0.0 s)
Depth 12 built in 0:00:00 (0.0 s)
Tree completed


In [8]:
print(root_node.get_value())

11.84047319901785


In [9]:
for lvl, node in iterate_nodes_by_levels(root_node):
    if not node.tree_completed():
        break

In [10]:
print(str(node.parent.bj_round))

Last action: HIT
Dealer 6,X
Player A,2,A,2,6,6(18)[$10]


In [11]:
node.tree_completed()

True

In [12]:
print(root_node.children_events)
print(root_node.children_prob)

[<PlayerAction.STAND: 'STAND'>, <PlayerAction.HIT: 'HIT'>, <PlayerAction.DOUBLE: 'DOUBLE'>, <PlayerAction.SPLIT: 'SPLIT'>]
[0, 0, 0, 1]


In [13]:
[ch.get_value() for ch in root_node.children]

[np.float64(-1.2),
 np.float64(8.6),
 np.float64(2.7181598062953998),
 np.float64(11.84047319901785)]

In [14]:
root_node.children_events

[<PlayerAction.STAND: 'STAND'>,
 <PlayerAction.HIT: 'HIT'>,
 <PlayerAction.DOUBLE: 'DOUBLE'>,
 <PlayerAction.SPLIT: 'SPLIT'>]

In [15]:
# Last card: 6
# Dealer 6X
# Player AA(12/2)[$10]
# Depth 0 built in 0:00:00 (0.0 s)
# Depth 1 built in 0:00:00 (0.0 s)
# Depth 2 built in 0:00:00 (0.0 s)
# Depth 3 built in 0:00:00 (0.0 s)
# Depth 4 built in 0:00:00 (0.0 s)
# Depth 5 built in 0:00:00 (0.0 s)
# Depth 6 built in 0:00:01 (1.0 s)
# Depth 7 built in 0:00:02 (2.0 s)
# Depth 8 built in 0:00:02 (2.0 s)
# Depth 9 built in 0:00:02 (2.0 s)
# Depth 10 built in 0:00:02 (2.0 s)
# Depth 11 built in 0:00:02 (2.0 s)
# Depth 12 built in 0:00:02 (2.0 s)
# Depth 13 built in 0:00:01 (1.0 s)
# Depth 14 built in 0:00:01 (1.0 s)
# Depth 15 built in 0:00:01 (1.0 s)
# Depth 16 built in 0:00:00 (0.0 s)
# Depth 17 built in 0:00:00 (0.0 s)
# Depth 18 built in 0:00:00 (0.0 s)
# Depth 19 built in 0:00:00 (0.0 s)
# Depth 20 built in 0:00:00 (0.0 s)
# Depth 21 built in 0:00:00 (0.0 s)
# Tree completed

In [16]:
log_tree_structure(root_node)

Total: 458 nodes
Level 0: 1 nodes Counter({'MixedNode_PLAYER_ACTION': 1})
Level 1: 4 nodes Counter({'ValueNode': 1, 'HitNode_PLAYER_CARD_FULL': 1, 'DoubleNode_PLAYER_CARD_FULL': 1, 'SplitNode_PLAYER_CARD_FULL': 1})
Level 2: 30 nodes Counter({'ValueNode': 12, 'DecisionNode_PENDING': 10, 'FloorCeilValueNode': 8})
Level 3: 29 nodes Counter({'ValueNode': 10, 'HitNode_PLAYER_CARD_FULL': 10, 'DoubleNode_PLAYER_CARD_FULL': 9})
Level 4: 190 nodes Counter({'ValueNode': 100, 'FloorCeilValueNode': 80, 'DecisionNode_PENDING': 8, 'DecisionNode_DECIDED': 2})
Level 5: 20 nodes Counter({'ValueNode': 10, 'HitNode_PLAYER_CARD_FULL': 10})
Level 6: 100 nodes Counter({'FloorCeilValueNode': 56, 'ValueNode': 40, 'DecisionNode_PENDING': 3, 'DecisionNode_DECIDED': 1})
Level 7: 8 nodes Counter({'ValueNode': 4, 'HitNode_PLAYER_CARD_FULL': 4})
Level 8: 40 nodes Counter({'FloorCeilValueNode': 25, 'ValueNode': 13, 'DecisionNode_PENDING': 1, 'DecisionNode_DECIDED': 1})
Level 9: 4 nodes Counter({'ValueNode': 2, 'HitN

In [17]:
for i in range(400):
    t0 = time.time()
    root_node.convert_to_full_up_to_depth(depth=i)
    t1 = time.time()
    seconds = np.round(t1 - t0)
    dt_whole = datetime.timedelta(seconds=seconds)
    print()
    print(f"Depth {i} converted to full enumeration in {dt_whole} ({seconds} s)")
    print("Value estimate = ", root_node.get_value())
    log_tree_structure(root_node)


Depth 0 converted to full enumeration in 0:00:00 (0.0 s)
Value estimate =  11.84047319901785
Total: 458 nodes
Level 0: 1 nodes Counter({'MixedNode_PLAYER_ACTION': 1})
Level 1: 4 nodes Counter({'ValueNode': 1, 'HitNode_PLAYER_CARD_FULL': 1, 'DoubleNode_PLAYER_CARD_FULL': 1, 'SplitNode_PLAYER_CARD_FULL': 1})
Level 2: 30 nodes Counter({'ValueNode': 12, 'DecisionNode_PENDING': 10, 'FloorCeilValueNode': 8})
Level 3: 29 nodes Counter({'ValueNode': 10, 'HitNode_PLAYER_CARD_FULL': 10, 'DoubleNode_PLAYER_CARD_FULL': 9})
Level 4: 190 nodes Counter({'ValueNode': 100, 'FloorCeilValueNode': 80, 'DecisionNode_PENDING': 8, 'DecisionNode_DECIDED': 2})
Level 5: 20 nodes Counter({'ValueNode': 10, 'HitNode_PLAYER_CARD_FULL': 10})
Level 6: 100 nodes Counter({'FloorCeilValueNode': 56, 'ValueNode': 40, 'DecisionNode_PENDING': 3, 'DecisionNode_DECIDED': 1})
Level 7: 8 nodes Counter({'ValueNode': 4, 'HitNode_PLAYER_CARD_FULL': 4})
Level 8: 40 nodes Counter({'FloorCeilValueNode': 25, 'ValueNode': 13, 'Decisio

KeyboardInterrupt: 

In [ ]:
print(root_node.get_value()) # 16.86824325912692

10.518987794797095


In [ ]:
lvl25 = None
for lvl, node in iterate_nodes_by_levels(root_node):
    if not node.tree_completed():
        incomplete = node
        break

In [ ]:
incomplete

In [ ]:
# with open("logs/game_tree.json", "w") as f:
#     game_tree_to_json(f, root_node)

In [ ]:
print(root_node.get_value())
print(root_node.children_prob)
print(root_node.children_events)
print([f"{ch.get_value():.2f}" for ch in root_node.children])

10.518987794797095
[0, 0, 0, 1]
[<PlayerAction.STAND: 'STAND'>, <PlayerAction.HIT: 'HIT'>, <PlayerAction.DOUBLE: 'DOUBLE'>, <PlayerAction.SPLIT: 'SPLIT'>]
['-1.20', '1.90', '2.72', '10.52']


In [ ]:
node = root_node.children[-1].children[7]
print("decision", node.decision_choice)
print(str(node.bj_round))
print("value = ", node.get_value())
print(node.children_prob)
print(node.children_events)

node.decision_action = None
for ch in node.children:
    ch.decision_action = None
    
print([f"{ch.get_value():.2f}" for ch in node.children])
print([f"{ch.get_ceil_value():.2f}" for ch in node.children])
print([f"{ch.get_floor_value():.2f}" for ch in node.children])

decision PlayerAction.STAND
Last card: 6
Dealer 6,X
Player A,9(10/20)[$10]
value =  6.9
[1, 0, 0]
[<PlayerAction.STAND: 'STAND'>, <PlayerAction.HIT: 'HIT'>, <PlayerAction.DOUBLE: 'DOUBLE'>]
['6.90', '2.89', '5.89']
['6.90', '3.57', '5.89']
['6.90', '2.89', '5.89']


In [ ]:
node.update_decision()

AttributeError: 'DecisionNode' object has no attribute 'update_decision'

In [ ]:
node.children[1].get_value()

np.float64(2.8939320388349516)

In [ ]:
print(node.children[1].cards_sampled) 
print(node.children[1].cards_not_sampled)
print(node.children[1].cards_21)
print(node.children[1].cards_bust)

[10]
[]
[11]
[]


In [ ]:
print(set(node.children[1].children_prob))

{0.07281553398058252, 0.3106796116504854, 0.07766990291262135, 0.07524271844660194}


In [ ]:
node_child = node.children[1]

print(node_child.children_prob)
print(node_child.children_events)
print([f"{ch.get_value():.2f}" for ch in node_child.children])


[0.07281553398058252, 0.3106796116504854, 0.07766990291262135, 0.07766990291262135, 0.07766990291262135, 0.07766990291262135, 0.07524271844660194, 0.07766990291262135, 0.07766990291262135, 0.07524271844660194]
[11, 10, 2, 3, 4, 5, 6, 7, 8, 9]
['8.90', '7.70', '-1.00', '-2.00', '-0.40', '-3.20', '-1.40', '0.00', '2.10', '4.10']


In [ ]:
np.array(node_child.children_prob).dot(np.array([ch.get_value() for ch in node_child.children]))

np.float64(2.8939320388349508)

In [ ]:
for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

PlayerAction.STAND
Total: 1 nodes
Level 0: 1 nodes Counter({'ValueNode': 1})
PlayerAction.HIT
Total: 179 nodes
Level 0: 1 nodes Counter({'HitNode_PLAYER_CARD_FULL': 1})
Level 1: 10 nodes Counter({'DecisionNode_PLAYER_ACTION': 9, 'ValueNode': 1})
Level 2: 18 nodes Counter({'ValueNode': 9, 'HitNode_PLAYER_CARD_FULL': 9})
Level 3: 90 nodes Counter({'ValueNode': 54, 'FloorCeilValueNode': 32, 'DecisionNode_PLAYER_ACTION': 4})
Level 4: 8 nodes Counter({'ValueNode': 4, 'HitNode_PLAYER_CARD_FULL': 4})
Level 5: 40 nodes Counter({'ValueNode': 28, 'FloorCeilValueNode': 11, 'DecisionNode_PLAYER_ACTION': 1})
Level 6: 2 nodes Counter({'ValueNode': 1, 'HitNode_PLAYER_CARD_FULL': 1})
Level 7: 10 nodes Counter({'ValueNode': 7, 'FloorCeilValueNode': 3})
PlayerAction.DOUBLE
Total: 11 nodes
Level 0: 1 nodes Counter({'DoubleNode_PLAYER_CARD_FULL': 1})
Level 1: 10 nodes Counter({'ValueNode': 10})


In [ ]:
node.rebuild_children()
node.build_tree()
node.convert_to_full_up_to_depth(np.inf)

0 [np.float64(-7.103649635036497)]


0 [np.float64(-4.376283618581907)]
0 [np.float64(-1.4048899755501219)]
0 [np.float64(-2.7995110024449885)]
0 [np.float64(-5.68117359413203)]
0 [np.float64(-7.271393643031786)]
0 [np.float64(-8.659902200488997)]
0 [np.float64(-2.828431372549019)]
0 [np.float64(-4.169117647058823)]
0 [np.float64(-5.762254901960784)]
0 [np.float64(-7.281372549019608)]
0 [np.float64(-8.709803921568627)]
0 [np.float64(-8.71965601965602)]
0 [np.float64(-1.6088235294117643)]
0 [np.float64(-3.050611246943766)]
1 [np.float64(-3.0)]
0 [np.float64(-5.6361858190709055)]
0 [np.float64(-2.799511002444988)]
0 [np.float64(-4.255990220048901)]
0 [np.float64(-7.183374083129585)]
0 [np.float64(-8.60635696821516)]
0 [np.float64(-4.438235294117646)]
0 [np.float64(-5.852941176470587)]
0 [np.float64(-7.237990196078432)]
0 [np.float64(-8.677941176470588)]
0 [np.float64(-2.7906862745098038)]
0 [np.float64(-3.6124694376528117)]
0 [np.float64(-3.013902439024391)]
0 [np.float64(-1.425365853658536)]
0 [np.float64(-7.13643031784841

True

In [ ]:
print(node.children_events)
print([f"{ch.get_value():.2f}" for ch in node.children])

for ch, ch_event in zip(node.children, node.children_events):
    print(ch_event)
    log_tree_structure(ch)

[<PlayerAction.STAND: 'STAND'>, <PlayerAction.HIT: 'HIT'>, <PlayerAction.DOUBLE: 'DOUBLE'>]
['6.90', '3.12', '5.79']
PlayerAction.STAND
Total: 1 nodes
Level 0: 1 nodes Counter({'ValueNode': 1})
PlayerAction.HIT
Total: 1043 nodes
Level 0: 1 nodes Counter({'HitNode_PLAYER_CARD_FULL': 1})
Level 1: 10 nodes Counter({'DecisionNode_PLAYER_ACTION': 9, 'ValueNode': 1})
Level 2: 18 nodes Counter({'ValueNode': 9, 'HitNode_PLAYER_CARD_FULL': 9})
Level 3: 90 nodes Counter({'ValueNode': 54, 'DecisionNode_PLAYER_ACTION': 27, 'FloorCeilValueNode': 9})
Level 4: 54 nodes Counter({'ValueNode': 27, 'HitNode_PLAYER_CARD_FULL': 27})
Level 5: 270 nodes Counter({'ValueNode': 194, 'FloorCeilValueNode': 47, 'DecisionNode_PLAYER_ACTION': 29})
Level 6: 58 nodes Counter({'ValueNode': 29, 'HitNode_PLAYER_CARD_FULL': 29})
Level 7: 290 nodes Counter({'ValueNode': 222, 'FloorCeilValueNode': 48, 'DecisionNode_PLAYER_ACTION': 20})
Level 8: 40 nodes Counter({'ValueNode': 20, 'HitNode_PLAYER_CARD_FULL': 20})
Level 9: 200

In [ ]:
root_node.convert_to_full_up_to_depth(depth=np.inf)

In [ ]:
values = []

values.append(root_node.get_value())

for i in tqdm.tqdm(range(1000)):
    root_node.resample_player_cards()
    values.append(root_node.get_value())
mean_value = np.mean(values)


  0%|          | 0/1000 [00:00<?, ?it/s]


AttributeError: 'MixedNode' object has no attribute 'resample_player_cards'

In [ ]:
f, ax = plt.subplots()
ax.scatter(range(len(values)), values)
ax.hlines(mean_value, xmin=0, xmax=len(values), color="red", label=f"mean={mean_value:.3f}")
ax.legend()

In [ ]:
np.min(values), np.max(values)

In [ ]:
for lvl in range(25):
    level_nodes = get_nodes_on_the_level(root_node, lvl)
    print(f"Level {lvl} has {len(level_nodes)} nodes")
    finished = False
    for n in level_nodes:
        if n.bj_round.get_stage() == BJStage.ROUND_OVER:
            continue
        elif isinstance(n, MonteCarloNode):
            print(f"MonteCarloNode found on level {lvl}")
            finished = True
            break
        else:
            break
    
    if finished:
        break

In [ ]:
from collections import Counter
count = Counter()

player_card_nodes = []
player_action_nodes = []
for node in level_nodes:
    stage = node.bj_round.get_stage()
    count[stage] += 1
    if stage == BJStage.PLAYER_CARD:
        player_card_nodes.append(node)
    if stage == BJStage.PLAYER_ACTION:
        player_action_nodes.append(node)



In [ ]:
count

In [ ]:
for i, n in enumerate(player_action_nodes):
    print(f"idx = {i}")
    print(f"Value = {n.get_value()}")
    print(str(n.bj_round))
    print()

In [ ]:
this_node = player_card_nodes[-1].parent
this_node.build_tree()

print(f"this_node Value = {this_node.get_value()}")
print(str(this_node.bj_round))
first_value = this_node.get_value()

In [ ]:
for i in range(100):
    changed = this_node.resample_player_cards()
    if this_node.get_value() == first_value:
        continue
    print(changed, this_node.get_value())
    print("-" * 32)

    # for i in range(len(this_node.children)):
    #     p = this_node.children_prob[i]
    #     ch = this_node.children[i]
    #     print(f"Value = {ch.get_value()}")
    #     print(f"Probability = {p}")
    #     print(str(ch.bj_round))
    #     print()
    # print("-" * 32)


In [ ]:
# this_node.has_completed_tree = False
# this_node.has_built_children = False
# this_node.children_prob = []
# this_node.children = []

In [ ]:
print(this_node.children[1].children[-1].bj_round)

In [ ]:
for i in range(100):
    this_node.resample_player_cards()
    children_of_interest = this_node.children[1].children[-1].children

    # if this_node.children_prob[1] == 0:
    #    continue
    
    # print(this_node.get_value())
    # print([f"{ch.get_value():.2f}" for ch in this_node.children])
    print(this_node.children_prob)

In [ ]:
this_node.resample_player_cards()

for ch in this_node.children:
    print(ch.get_value())
    print(str(ch.bj_round))
    print()

In [ ]:

print(this_node.children[0].get_value())

In [ ]:
print(this_node.children)